In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# Milestone 1

In [3]:
print(train.columns)

Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object')


In [4]:
# Answer 1
freq = train['answer'].value_counts()
print(freq)
print(f'Most Frequent: {freq.max()}')
print(f'least Frequent: {freq.min()}')
print("Sum:", freq.min() + freq.max())

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Most Frequent: 490
least Frequent: 324
Sum: 814


In [5]:
# Answer 2
import string
prompt_lowercase = train["prompt"].str.lower()

prompts = prompt_lowercase.str.translate(str.maketrans('', '', string.punctuation))

vocab = set()

for text in prompts:
    vocab.update(text.split())

print(len(vocab))

859


In [6]:
# Answer 3
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import string

prompt = train.loc[train["id"] == 1, "prompt"].iloc[0]

prompt = prompt.lower()

prompt = prompt.translate(
    str.maketrans('', '', string.punctuation)
)

words = prompt.split()

filtered_words = [w for w in words if w not in ENGLISH_STOP_WORDS]

print(len(filtered_words))

13


In [7]:
# Answer 4
from sklearn.feature_extraction.text import TfidfVectorizer

combined_text = (
    
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(combined_text)

print(X.shape[1])

2762


In [8]:
# Answer 5
from sklearn.metrics.pairwise import cosine_similarity

row = train[train["id"] == 1].iloc[0]

prompt_vec = vectorizer.transform([row["prompt"]])
option_a_vec = vectorizer.transform([row["A"]])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]

print(round(similarity, 4))

0.272


In [9]:
# Answer 6
from sklearn.metrics.pairwise import cosine_similarity

correct = 0
total = len(train)

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    sims = {}

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sims[option] = cosine_similarity(prompt_vec, option_vec)[0][0]

    predicted = max(sims, key=sims.get)

    if predicted == row["answer"]:
        correct += 1

accuracy = (correct / total) * 100

print(round(accuracy, 2))

13.55


In [10]:
# ans 7. 1.0
# ans 8 0.5
# ans 9 0.4213



In [11]:
# ansswer 10
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

def map3_score(actual, preds):
    if actual in preds[:3]:
        return 1 / (preds.index(actual) + 1)
    return 0

scores = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    sims = {}

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sims[option] = cosine_similarity(prompt_vec, option_vec)[0][0]

    preds = sorted(sims, key=sims.get, reverse=True)

    scores.append(map3_score(row["answer"], preds))

print(round(sum(scores) / len(scores), 5))

0.29617


# milestone-2

In [12]:
#q1
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]
dataset = dataset.map(
    lambda x: {
        "combined_text": x["prompt"] + " " + x["A"]
    }
)
text = dataset[51]["combined_text"]
print(text)
length = len(text)
print(length)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
614


In [13]:
#q2
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


In [14]:
#q3
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

[SEP]
102


In [15]:
#q4
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

prompts = list(dataset["prompt"])   

encodings = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([2000, 128])


In [16]:
#q5
#64

In [17]:
#q6
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

text = dataset[0]["prompt"]

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


In [18]:
#q7
cls_embedding = outputs.last_hidden_state[0, 0]

result = cls_embedding[:5].sum().item()

print(round(result, 4))

-1.2001


In [19]:
#q7
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)


tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
for i, token in enumerate(tokens):
    print(i, token)


fusion_index = tokens.index("fusion")


attention = outputs.attentions[-1][0, 0]


value = attention[0, fusion_index].item()

print("Attention:", round(value, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]
Attention: 0.1025


In [20]:
#Q8
from sentence_transformers import SentenceTransformer, util


model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]


prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)


similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print(round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658


In [21]:
#q9
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

options = ["A", "B", "C", "D", "E"]

mini_hits = 0
tfidf_hits = 0
improved = 0

for row in dataset:
    prompt = row["prompt"]
    answer = row["answer"]   


    texts = [prompt] + [row[o] for o in options]
    tfidf = TfidfVectorizer().fit_transform(texts)
    sims = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
    tfidf_top3 = [options[i] for i in np.argsort(sims)[::-1][:3]]


    embeddings = model.encode(texts, convert_to_tensor=True)
    sims = util.cos_sim(embeddings[0], embeddings[1:]).cpu().numpy().flatten()
    minilm_top3 = [options[i] for i in np.argsort(sims)[::-1][:3]]


    if answer in minilm_top3:
        rank = minilm_top3.index(answer) + 1
        mini_hits += 1 / rank


    if answer not in tfidf_top3 and answer in minilm_top3:
        improved += 1

map3 = mini_hits / len(dataset)

print("MAP@3:", round(map3, 4))
print("Improved count:", improved)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MAP@3: 0.4231
Improved count: 564


In [22]:
#q10
from transformers import pipeline

classifier = pipeline("zero-shot-classification")

text = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"],
]

result = classifier(text, candidate_labels)

print(result["labels"][0])
print(round(result["scores"][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.
0.4575


In [23]:
#q11
result_multi = classifier(
    text,
    candidate_labels,
    multi_label=True
)

print(result_multi["scores"])
print(sum(result_multi["scores"]))

[0.00046927088988013566, 2.0635825421777554e-05, 1.970058110600803e-05]
0.0005096072964079212


In [24]:
#Q12
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row = dataset[0]

text = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=5)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

B
